# Step 01: Dataset Explorer & 3 Representation Visualizations

Explores raw mmWave radar point cloud motion trajectories and visually renders sample data across **all 3 representations** side-by-side:
1. **Representation 1 (`rep1_doppler`)**: Micro-Doppler Velocity-Time Spectrogram Map.
2. **Representation 2 (`rep2_projections`)**: 2D Orthogonal Spatial Occupancy Projections (Elevation, Ground, Doppler-Height).
3. **Representation 3 (`rep3_pointset`)**: Native 3D Centered Point Cloud Tensor.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(root_dir) not in sys.path: sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset_parser import get_all_recording_files, load_recording_df, extract_label
from src.transforms import (center_target_relative, extract_raw_windows_from_df,
                            transform_rep1_doppler_time, transform_rep2_orthogonal_projections,
                            transform_rep3_pointnet)

sns.set_theme(style="whitegrid")


In [ ]:
# Load sample Fall recording and sample ADL recording
all_files = get_all_recording_files()
sample_fall_f = [f for f in all_files if extract_label(f) == 1][0]
sample_adl_f = [f for f in all_files if extract_label(f) == 0][0]

df_fall_raw = load_recording_df(sample_fall_f)
df_adl_raw = load_recording_df(sample_adl_f)

# Extract raw 10-frame windows
fall_wins = extract_raw_windows_from_df(df_fall_raw, window_size_frames=10, stride=5)
sample_win = fall_wins[0:1]  # Shape: (1, 10, 32, 4)

# Generate all 3 representations for the sample window
rep1_img = transform_rep1_doppler_time(sample_win)[0]         # (3, 32, 32)
rep2_img = transform_rep2_orthogonal_projections(sample_win)[0] # (3, 32, 32)
rep3_tensor = sample_win[0]                                   # (10, 32, 4)

print(f"Sample Recording: {sample_fall_f.name}")
print(f"Rep 1 Shape: {rep1_img.shape} | Rep 2 Shape: {rep2_img.shape} | Rep 3 Shape: {rep3_tensor.shape}")


In [ ]:
# Showcase Data in 3 Different Presentations Side-by-Side
fig = plt.figure(figsize=(18, 5), dpi=100)

# Presentation 1: Micro-Doppler Velocity-Time Map (Rep 1)
ax1 = fig.add_subplot(1, 4, 1)
ax1.imshow(rep1_img[0], cmap='inferno', aspect='auto', origin='lower')
ax1.set_title("1. Micro-Doppler Map (Rep 1)\n(Velocity vs Time Grid)", fontweight='bold', fontsize=11)
ax1.set_xlabel("Time Bin"); ax1.set_ylabel("Doppler Velocity Bin")

# Presentation 2: Elevation Projection (Rep 2 Channel 0)
ax2 = fig.add_subplot(1, 4, 2)
ax2.imshow(rep2_img[0], cmap='viridis', aspect='auto', origin='lower')
ax2.set_title("2. Elevation Projection (Rep 2)\n(Delta X vs Z Occupancy)", fontweight='bold', fontsize=11)
ax2.set_xlabel("Delta X Bin"); ax2.set_ylabel("Elevation Z Bin")

# Presentation 2: Ground Projection (Rep 2 Channel 1)
ax3 = fig.add_subplot(1, 4, 3)
ax3.imshow(rep2_img[1], cmap='plasma', aspect='auto', origin='lower')
ax3.set_title("2. Ground Projection (Rep 2)\n(Delta X vs Delta Y Occupancy)", fontweight='bold', fontsize=11)
ax3.set_xlabel("Delta X Bin"); ax3.set_ylabel("Delta Y Bin")

# Presentation 3: Native 3D Centered Point Cloud (Rep 3)
ax4 = fig.add_subplot(1, 4, 4, projection='3d')
pts = rep3_tensor.reshape(-1, 4)
sc = ax4.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=pts[:, 3], cmap='coolwarm', s=35, edgecolors='black')
ax4.set_title("3. Native 3D Point Set (Rep 3)\n(Delta X, Delta Y, Z colored by v)", fontweight='bold', fontsize=11)
ax4.set_xlabel("Delta X"); ax4.set_ylabel("Delta Y"); ax4.set_zlabel("Elevation Z")
plt.colorbar(sc, ax=ax4, shrink=0.5, label='Doppler v (m/s)')

plt.tight_layout()
plt.show()
